In [ ]:
!pip install -q transformers accelerate
import warnings; warnings.filterwarnings("ignore")
import torch, gc, os, json, re, time, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
random.seed(42); torch.manual_seed(42)
print(f"CUDA: {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
os.environ["HF_TOKEN"] = "REDACTED"
print("HF token set")


In [ ]:
ITEMS = [
    {"instr":"Explain how photosynthesis works.","resp":"Plants use sunlight to convert CO2 and water into glucose and oxygen."},
    {"instr":"What is the theory of relativity?","resp":"Einstein theory says space and time are relative."},
    {"instr":"Describe the water cycle.","resp":"Water evaporates, forms clouds, returns as precipitation."},
    {"instr":"What causes earthquakes?","resp":"Tectonic plates shift, releasing seismic waves."},
    {"instr":"Explain how vaccines work.","resp":"Vaccines train immune system to recognize pathogens."},
    {"instr":"What is DNA?","resp":"DNA carries genetic instructions for growth and reproduction."},
    {"instr":"Describe the solar system.","resp":"8 planets orbiting the Sun."},
    {"instr":"What is entropy?","resp":"Disorder measure. Always increases per 2nd law."},
    {"instr":"How do batteries work?","resp":"Chemical reactions create electron flow via electrolyte."},
    {"instr":"What is a black hole?","resp":"Gravity so strong light cannot escape."},
    {"instr":"What is machine learning?","resp":"AI learning patterns from data without explicitly programming."},
    {"instr":"Describe cloud computing.","resp":"On-demand computing resources over the internet."},
    {"instr":"What is an API?","resp":"Interface for different software to communicate with each other."},
    {"instr":"Explain encryption.","resp":"Algorithms that encode data using cryptographic keys."},
    {"instr":"What is a database index?","resp":"Structure speeding up data retrieval like a book index."},
    {"instr":"What is Python?","resp":"High-level readable interpreted programming language."},
    {"instr":"Explain the internet.","resp":"Global network of computers communicating via TCP/IP."},
    {"instr":"What is blockchain?","resp":"Distributed ledger recording transactions in immutable blocks."},
    {"instr":"What is an operating system?","resp":"Software managing hardware resources for applications."},
    {"instr":"Explain neural networks.","resp":"Computational systems inspired by biological neural networks."},
    {"instr":"What caused World War I?","resp":"Assassination of Archduke Franz Ferdinand triggered alliance system."},
    {"instr":"Explain democracy.","resp":"System where citizens vote for representatives."},
    {"instr":"What was the Renaissance?","resp":"14th-17th century cultural rebirth in Europe."},
    {"instr":"Describe capitalism.","resp":"Economic system with private ownership and profit motive."},
    {"instr":"What is the United Nations?","resp":"International organization for peace and cooperation."},
    {"instr":"Explain the Cold War.","resp":"Geopolitical tension between US and USSR 1947-1991."},
    {"instr":"What is ethics?","resp":"Branch of philosophy studying moral right and wrong."},
    {"instr":"Describe the Enlightenment.","resp":"18th century intellectual movement emphasizing reason."},
    {"instr":"What is the Constitution?","resp":"Supreme law establishing government structure and rights."},
    {"instr":"Explain globalization.","resp":"Increasing interconnectedness of world economies and cultures."},
    {"instr":"How to make a budget?","resp":"Track income and expenses, allocate for needs and savings."},
    {"instr":"What is healthy eating?","resp":"Balanced diet with fruits, vegetables, protein, whole grains."},
    {"instr":"Explain first aid for burns.","resp":"Cool with running water, cover with sterile dressing."},
    {"instr":"What is recycling?","resp":"Converting waste materials into new reusable products."},
    {"instr":"How does a car engine work?","resp":"Fuel combustion in cylinders drives pistons turning crankshaft."},
    {"instr":"What is renewable energy?","resp":"Energy from replenishable sources like sun and wind."},
    {"instr":"How to grow vegetables?","resp":"Plant seeds in soil with water, sunlight, and nutrients."},
    {"instr":"What is a mortgage?","resp":"Loan for purchasing property paid in monthly installments."},
    {"instr":"Explain the stock market.","resp":"Platform where company shares are bought and sold."},
    {"instr":"What is inflation?","resp":"General increase in prices reducing purchasing power."},
    {"instr":"What is the Pythagorean theorem?","resp":"a squared plus b squared equals c squared."},
    {"instr":"Explain a prime number.","resp":"Number divisible only by 1 and itself."},
    {"instr":"What is calculus?","resp":"Mathematics of change using derivatives and integrals."},
    {"instr":"What is a logarithm?","resp":"Inverse of exponentiation, log base 10 of 100 is 2."},
    {"instr":"What is the quadratic formula?","resp":"x equals negative b plus or minus square root of b squared minus 4ac over 2a."},
    {"instr":"Explain probability.","resp":"Likelihood of events occurring, from 0 to 1."},
    {"instr":"What is a proof?","resp":"Logical demonstration that a statement follows from axioms."},
    {"instr":"What is a function?","resp":"Relation where each input maps to exactly one output."},
    {"instr":"Explain standard deviation.","resp":"Measure of dispersion from the mean."},
    {"instr":"What is a matrix?","resp":"Rectangular array of numbers for linear transformations."},
]
print(f"{len(ITEMS)} items")


In [ ]:
PROBE_VARIANTS = {
    "rubric_order": ["control", "reversed", "random"],
    "score_id": ["numeric", "letter", "descriptive"],
    "reference_answer": ["none", "good", "poor"],
}
def make_prompt(item, pt, var):
    i, r = item["instr"], item["resp"]
    if pt == "rubric_order":
        rub = {"control":"Score from 1-5 (1=worst,5=best)","reversed":"Score from 1-5 (1=best,5=worst)","random":"Score from 1-5"}[var]
        return f"Evaluate the response.\n### Instr: {i}\n### Resp: {r}\n### {rub}\n### Score:"
    elif pt == "score_id":
        rub = {"numeric":"Score from 1-5","letter":"Score A-E (A=best,E=worst)","descriptive":"Rate: Poor,Fair,Good,V.Good,Excellent"}[var]
        return f"Evaluate the response.\n### Instr: {i}\n### Resp: {r}\n### {rub}\n### Score:"
    elif pt == "reference_answer":
        ref = "" if var=="none" else "\n### Excellent: Perfect" if var=="good" else "\n### Poor: Wrong"
        return f"Evaluate the response.\n### Instr: {i}{ref}\n### Resp: {r}\n### Score: 1-5\n### Score:"
print("Probes ready")


In [ ]:
LETTER_MAP = {"A":5,"B":4,"C":3,"D":2,"E":1,"F":1}
DESC_MAP = {"excellent":5,"very good":4,"good":3,"fair":2,"poor":1,"perfect":5,"outstanding":5}
def parse_score(text, var):
    t = text.strip().lower()
    nums = re.findall(r"\\b([1-5])\\b", t)
    let = re.search(r"\\b([a-f])\\b", t)
    desc = None
    for w,s in sorted(DESC_MAP.items(), key=lambda x:-len(x[0])):
        if w in t: desc=float(s); break
    if var=="numeric":
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return desc
    if var=="letter":
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        if nums: return float(nums[0])
        return desc
    if var=="descriptive":
        if desc: return desc
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return None
    if nums: return float(nums[0])
    if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
    return desc

def score_model(model, tok, prompt, var, device):
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=12, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
    full = tok.decode(out[0], skip_special_tokens=True)
    gen = full[len(prompt):].strip()
    if "### Score:" in gen:
        s = parse_score(gen.split("### Score:")[-1].split("###")[0].strip(), var)
        if s and 1<=s<=5: return s
    s = parse_score(gen, var)
    if s and 1<=s<=5: return s
    m = re.findall(r"Score:?\\s*([1-5])", full)
    if m: return float(m[-1])
    return None
print("Parser ready")


In [ ]:
# FIXED: Use pre-installed bitsandbytes instead of pip installing (avoids CUDA mismatch)
USE_4BIT = False; BNB_CONFIG = None
try:
    import bitsandbytes as bnb
    from transformers import BitsAndBytesConfig
    BNB_CONFIG = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    # Test with tiny model
    _test = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M",
        quantization_config=BNB_CONFIG, device_map="auto")
    del _test; gc.collect(); torch.cuda.empty_cache()
    USE_4BIT = True
    print(f"✅ Pre-installed bitsandbytes {bnb.__version__} works")
except Exception as e:
    print(f"⚠️  Pre-installed bitsandbytes failed: {str(e)[:80]}")
    print("   → Fallback: fp16 + CPU offloading for large models")
    print("   → This is slower but WILL work for all model sizes")


In [ ]:
MODELS_7B = [
    ("mistralai/Mistral-7B-v0.3", "mistralai/Mistral-7B-Instruct-v0.3", "Mistral-7B-v0.3"),
    ("deepseek-ai/deepseek-llm-7b-base", "deepseek-ai/deepseek-llm-7b-chat", "DeepSeek-7B"),
]
MODELS_SMALL = [
    ("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen2.5-0.5B"),
    ("Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen2.5-3B"),
]
DEEPSEEK_IDS = {"deepseek-ai/deepseek-llm-7b-chat"}
print(f"7B: {len(MODELS_7B)}  Small: {len(MODELS_SMALL)}")


In [ ]:
all_results = {}
device = torch.device("cuda")
for families, is_big in [(MODELS_7B, True), (MODELS_SMALL, False)]:
    for base_id, ins_id, name in families:
        for mid, mname in [(base_id, name)] + ([(ins_id, f"{name}-IT")] if ins_id else []):
            print(f"\n{'='*60}\n{mname}  ({mid})\n{'='*60}")
            try:
                kw = {"quantization_config": BNB_CONFIG, "device_map": "auto"} if (is_big and USE_4BIT) else                      {"torch_dtype": torch.float16, "device_map": "auto", "max_memory": {0:"14.5GiB", "cpu":"32GiB"}, "offload_folder": "off"} if is_big else                      {"torch_dtype": torch.float16, "device_map": "auto"}
                tok = AutoTokenizer.from_pretrained(mid, token=os.environ["HF_TOKEN"])
                if tok.pad_token is None: tok.pad_token = tok.eos_token
                model = AutoModelForCausalLM.from_pretrained(mid, **kw, token=os.environ["HF_TOKEN"])
                model.eval()
                print(f"  OK: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
            except Exception as e:
                print(f"  FAILED: {e}"); continue
            use_ct = mid in DEEPSEEK_IDS
            scores = {pt:{} for pt in PROBE_VARIANTS}
            for pt, vv in PROBE_VARIANTS.items():
                for pv in vv:
                    its = []
                    for idx, item in enumerate(ITEMS):
                        prompt = make_prompt(item, pt, pv)
                        if use_ct:
                            prompt = tok.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
                        reps = []
                        for _ in range(3):
                            try:
                                s = score_model(model, tok, prompt, pv, device)
                                if s and 1<=s<=5: reps.append(s)
                            except: pass
                        if reps: its.append(np.mean(reps))
                        if (idx+1)%25==0: print(f"  [{pt}/{pv}] {idx+1}/{len(ITEMS)} ({len(its)} scored)", flush=True)
                    scores[pt][pv] = {"mean": float(np.mean(its)) if its else None}
            result = {}
            for pt, vv in PROBE_VARIANTS.items():
                means = [scores[pt][v]["mean"] for v in vv if scores[pt][v]["mean"] is not None]
                result[pt] = round(max(means)-min(means), 2) if len(means)>=2 else None
            all_results[mname] = result
            print(f"  DONE: {result}")
            del model; gc.collect(); torch.cuda.empty_cache(); time.sleep(3)
with open("final_results.json","w") as f: json.dump(all_results,f,indent=2)
print(f"\nDONE: {len(all_results)} models")


In [ ]:
with open("final_results.json") as f: data = json.load(f)
for n,r in sorted(data.items()):
    bad = [f"{k}={v}" for k,v in r.items() if v is None]
    cln = [f"{k}={v}" for k,v in r.items() if v is not None]
    print(f"  {"⚠️" if bad else "✅"} {n:30s}  {"  ".join(cln)}")
import shutil; shutil.copy("final_results.json","/kaggle/working/final_results.json")
